# Discretization and Binning in Feature Engineering

## 1. Clear Overview

Discretization, or binning, is a data preprocessing technique that converts a continuous numerical variable into a set of discrete intervals or categories. By transforming continuous data into bins, such as mapping ages into categories like `Young`, `Adult`, and `Senior`, practitioners can simplify complex data, identify nonlinear relationships, and reduce the impact of noise and extreme outliers.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: All necessary libraries successfully imported!")

## 2. Structured Table of Contents

- **Synthetic Data Creation**: Building our robust testing environment
- **Purpose of Binning**: Why we transition from continuous to discrete
- **Core Concept 1**: Equal Width Binning
- **Core Concept 2**: Equal Frequency Binning (Quantile)
- **Core Concept 3**: Custom Domain-Specific Binning
- **Scikit-Learn Implementation**: Safe pipelines without data leakage
- **Practice Exercises**: Apply what you've learned
- **Summary**: Key takeaways

Before diving into the methodology, let's generate a synthetic banking dataset. We will introduce deliberate statistical structures (like skewness and outliers) to demonstrate exactly why binning is necessary.

In [ ]:
# Step 2: Create synthetic banking dataset

# Generate a skewed age distribution to demonstrate binning challenges
ages_main = np.random.normal(loc=35, scale=8, size=800)      # Middle-aged majority
ages_senior = np.random.normal(loc=70, scale=6, size=150)    # Senior tail
ages_young = np.random.normal(loc=20, scale=2, size=50)      # Young minority

# Combine and clip to realistic bank customer ages
all_ages = np.concatenate([ages_main, ages_senior, ages_young])
all_ages = np.clip(all_ages, 18, 90).round(1)

# Generate an exponentially distributed balance feature (high skew)
balances = np.random.exponential(scale=2000, size=1000).round(2)

# Target variable: subscribed to term deposit (1 = Yes, 0 = No)
# Introduce a non-linear relationship: older people and higher balance people subscribe more
prob_subscribe = (all_ages / 200) + (balances / 20000)
prob_subscribe = np.clip(prob_subscribe, 0.05, 0.95)
targets = np.random.binomial(n=1, p=prob_subscribe)

# Assemble DataFrame
df = pd.DataFrame({
    'age': all_ages,
    'balance': balances,
    'subscribed': targets
})

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Synthetic Banking Dataset Created!\n")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

## 3. Purpose of Binning

While continuous variables provide granular information, binning allows for a more robust representation of data by:

1. **Simplifying Complex Data:** Grouping values reduces variance and highlights broader macro-trends.
2. **Handling Nonlinearity:** It enables linear machine learning models to identify nonlinear relationships by assigning distinct coefficients to different categorical bins.
3. **Improving Robustness:** Binning minimizes the destructive influence of small fluctuations, noise, and extreme outliers.
4. **Enhancing Interpretability:** Categorical labels (like `Low`, `Medium`, `High`) are often more intuitive for human analysis.

Let's visualize the raw distribution of our `age` variable.

In [ ]:
# Visualize raw distribution showing why binning helps simplify complex data
plt.figure(figsize=(10, 5))
sns.histplot(df['age'], bins=50, kde=True, color='skyblue')

plt.title('Distribution of Age in Banking Dataset', fontsize=14)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)

plt.axvline(df['age'].mean(), color='red', linestyle='--', label=f'Mean: {df["age"].mean():.1f}')
plt.axvline(df['age'].median(), color='green', linestyle='-', label=f'Median: {df["age"].median():.1f}')

plt.legend()
plt.tight_layout()
plt.show()

print("Diagnostic: Notice the multi-modal nature and the long right tail (seniors). Binning can simplify this.")

## 4. Core Concept 1: Equal Width Binning

In **Equal Width Binning**, the continuous range is divided into a specified number of intervals, each having the exact same range size. 
Formula: Width = (Max - Min) / Number of Bins

- **Strength:** Simple, mathematically straightforward, and easy to interpret boundaries.
- **Weakness:** It completely ignores data density. If you have extreme outliers, you might end up with bins that contain zero or very few observations.

In [ ]:
# Apply Equal Width Binning using Pandas cut()
# retbins=True allows us to capture the exact mathematical boundaries generated
df['age_bin_width'], width_edges = pd.cut(
    df['age'], 
    bins=4, 
    labels=['Young', 'Adult', 'Mature', 'Senior'],
    retbins=True
)

print("Equal Width Mathematical Edges:", width_edges.round(1))
print("\nObservation Count per Equal Width Bin:")
print(df['age_bin_width'].value_counts(sort=False))

print("\nInsight: Because our data is concentrated around 35, the 'Adult' bin is massive, while 'Mature' is sparse.")

### Visualizing Equal Width Boundaries
Let's map these mathematically generated boundaries over our raw distribution to see the effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Histogram with vertical lines for bin edges
sns.histplot(df['age'], bins=50, color='lightgray', ax=axes[0])
for edge in width_edges:
    axes[0].axvline(edge, color='red', linestyle='--', alpha=0.8)
axes[0].set_title('Equal Width Bin Edges Overlay', fontsize=14)
axes[0].set_xlabel('Age')

# Plot 2: Resulting population per bin
sns.countplot(data=df, x='age_bin_width', palette='Reds_d', ax=axes[1])
axes[1].set_title('Population per Equal Width Bin', fontsize=14)
axes[1].set_xlabel('Age Category')
axes[1].set_ylabel('Observation Count')

plt.tight_layout()
plt.show()

## 5. Core Concept 2: Equal Frequency Binning (Quantile Binning)

In **Equal Frequency Binning**, the range is divided such that each bin contains approximately the same number of data points. This relies on statistical quantiles/percentiles.

- **Strength:** Effectively balances the distribution of samples. It is extremely robust to outliers and skewed datasets.
- **Weakness:** Bins will have varying width ranges. One bin might cover 5 years, while another covers 30 years, which can be harder to explain to business stakeholders.

In [ ]:
# Apply Equal Frequency Binning using Pandas qcut()
df['age_bin_freq'], freq_edges = pd.qcut(
    df['age'], 
    q=4, 
    labels=['Q1', 'Q2', 'Q3', 'Q4'],
    retbins=True
)

print("Equal Frequency Bin Edges (Quantiles):", freq_edges.round(1))
print("\nObservation Count per Equal Frequency Bin:")
print(df['age_bin_freq'].value_counts(sort=False))

print("\nInsight: Notice how perfectly balanced the counts are. Each bin holds exactly 250 records.")

### Visualizing Equal Frequency Boundaries
Notice how the boundaries contract in dense regions and expand in sparse regions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Histogram with quantile edges
sns.histplot(df['age'], bins=50, color='lightgray', ax=axes[0])
for edge in freq_edges:
    axes[0].axvline(edge, color='green', linestyle='--', alpha=0.8)
axes[0].set_title('Equal Frequency Bin Edges Overlay', fontsize=14)
axes[0].set_xlabel('Age')

# Plot 2: Balanced bar plot
sns.countplot(data=df, x='age_bin_freq', palette='Greens_d', ax=axes[1])
axes[1].set_title('Population per Equal Frequency Bin', fontsize=14)
axes[1].set_xlabel('Quantile Category')
axes[1].set_ylabel('Observation Count')

plt.tight_layout()
plt.show()

## 6. Core Concept 3: Custom Domain-Specific Binning

Often, strictly mathematical approaches lack business sense. Business logic frequently dictates standard groupings (e.g., typical risk brackets, legal age limits).

We can apply domain knowledge using Pandas `cut` with explicit threshold lists, or `np.where` for rapid binary flagging.

In [ ]:
# Strategy A: Binary Flagging for specific high-value thresholds
# E.g., Creating a quick flag to check if the person is a senior
df['is_senior'] = np.where(df['age'] > 60, 'Yes', 'No')

# Strategy B: Custom Business Logic Bins
# Youth (18-25), Professional (25-45), Transition (45-60), Retired (60-100)
custom_bins = [18, 25, 45, 60, 100]
custom_labels = ['Youth', 'Professional', 'Transition', 'Retired']

df['age_custom'] = pd.cut(
    df['age'], 
    bins=custom_bins, 
    labels=custom_labels, 
    include_lowest=True
)

print("Custom Domain Bin Counts:")
print(df['age_custom'].value_counts(sort=False))

print("\nBinary Senior Flag Counts:")
print(df['is_senior'].value_counts())

## 7. Comparative Target Analysis

Why do we bin? To help models detect signal. Let's group by our custom bins and calculate the mean subscription rate to reveal the underlying nonlinear relationship.

In [ ]:
# Calculate conversion (subscription) rate per custom age group
conversion_rates = df.groupby('age_custom', observed=True)['subscribed'].mean().reset_index()
conversion_rates['subscribed_pct'] = (conversion_rates['subscribed'] * 100).round(1)

print("Subscription Rates by Custom Age Group:")
print(conversion_rates[['age_custom', 'subscribed_pct']])

# Visualize the signal extraction
plt.figure(figsize=(8, 5))
sns.barplot(data=conversion_rates, x='age_custom', y='subscribed_pct', palette='viridis')
plt.title('Subscription Rate by Age Category', fontsize=14)
plt.xlabel('Business Age Group')
plt.ylabel('Subscription Rate (%)')

# Draw a line for the global average to baseline the visualization
plt.axhline(df['subscribed'].mean() * 100, color='red', linestyle='--', label='Global Average')
plt.legend()
plt.tight_layout()
plt.show()

print("Takeaway: Binning clearly maps out a non-linear relationship. Retired customers convert at a vastly higher rate.")

## 8. Visualization Gallery

To summarize the differences, let's plot the raw data alongside our three automated/semi-automated binning structures in a single matrix.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-Left: Raw Distribution
sns.histplot(df['age'], kde=True, ax=axes[0, 0], color='gray')
axes[0, 0].set_title('Raw Continuous Age Distribution', fontsize=14)

# Top-Right: Equal Width
sns.countplot(data=df, x='age_bin_width', palette='Reds', ax=axes[0, 1])
axes[0, 1].set_title('Equal Width Binning (Fixed Interval)', fontsize=14)

# Bottom-Left: Equal Frequency
sns.countplot(data=df, x='age_bin_freq', palette='Greens', ax=axes[1, 0])
axes[1, 0].set_title('Equal Frequency Binning (Quantile)', fontsize=14)

# Bottom-Right: Domain Specific
sns.countplot(data=df, x='age_custom', palette='Purples', ax=axes[1, 1])
axes[1, 1].set_title('Custom Business Logic Binning', fontsize=14)

plt.suptitle('Comprehensive Gallery of Binning Strategies', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

## 9. Scikit-Learn Implementation & Avoiding Data Leakage

> **Warning:** A common pitfall in machine learning is calculating quantiles on the entire dataset *before* train/test splitting. This leaks target and distribution knowledge to the model.

While Pandas (`qcut`, `cut`) is excellent for exploratory analysis, production ML pipelines should utilize Scikit-Learn's `KBinsDiscretizer` to strictly learn boundaries on training data and subsequently map those to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer

# 1. Strictly Split the Data First
X_train, X_test, y_train, y_test = train_test_split(
    df[['age']], df['subscribed'], test_size=0.2, random_state=42
)

# 2. Initialize Discretizer (strategy='quantile' replicates pd.qcut)
# encode='ordinal' returns integers representing the bin indices
discretizer = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')

# 3. Fit strictly on train, transform both
X_train_binned = discretizer.fit_transform(X_train)
X_test_binned = discretizer.transform(X_test)

print("Scikit-learn KBinsDiscretizer applied successfully!")
print(f"Train arrays shape: {X_train_binned.shape}, Test arrays shape: {X_test_binned.shape}")

In [ ]:
# Inspect the learned boundaries from Scikit-Learn
sklearn_edges = discretizer.bin_edges_[0]

print("Scikit-Learn Quantile Boundaries (Derived from Train Only):")
for i in range(len(sklearn_edges)-1):
    print(f"Bin {i}: {sklearn_edges[i]:.1f} to {sklearn_edges[i+1]:.1f}")

## 10. Practice Exercises

Now it is your turn. We will shift focus to the `balance` feature, which is highly skewed (exponentially distributed) and contains extreme outliers.

### Exercise 1: Handling Highly Skewed Features

**Task:** 
1. Create 5 equal-width bins for the `balance` column using Pandas.
2. Create 5 equal-frequency bins for the `balance` column using Pandas.
3. Print the counts to see the catastrophic impact outliers have on Equal Width binning.

In [ ]:
# --- EXERCISE 1 SOLUTION ---

# 1. Equal Width (5 bins)
df['balance_width'] = pd.cut(df['balance'], bins=5)

# 2. Equal Frequency (5 bins)
df['balance_freq'] = pd.qcut(df['balance'], q=5)

print("--- Equal Width Balance Observation Counts ---")
print(df['balance_width'].value_counts(sort=False))

print("\n--- Equal Frequency Balance Observation Counts ---")
print(df['balance_freq'].value_counts(sort=False))

print("\nKey Observation: Equal Width completely fails here. Outliers stretch the maximum boundary so far that almost all data falls into the first bin!")

### Exercise 2: Visualizing the Exercise Results

**Task:** Create a side-by-side seaborn `countplot` comparing the bin populations of your new `balance_width` and `balance_freq` features.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plotting Equal Width
sns.countplot(data=df, y='balance_width', ax=axes[0], palette='Reds')
axes[0].set_title('Balance: Equal Width Failure', fontsize=14)
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Balance Ranges')

# Plotting Equal Frequency
sns.countplot(data=df, y='balance_freq', ax=axes[1], palette='Oranges')
axes[1].set_title('Balance: Equal Frequency Success', fontsize=14)
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Balance Quantiles')

plt.tight_layout()
plt.show()

## 11. Application Summary & Key Takeaways

- **Information Trade-off:** While binning simplifies data and treats noise, it involves a fundamental loss of precision by grouping distinct continuous values into generic categories.
- **Model Sensitivity:** 
  - *Equal Width* is strictly governed by Data Max and Min. It is highly vulnerable to outliers.
  - *Equal Frequency (Quantile)* is governed by density. It gracefully handles skewness and nullifies the impact of extreme outliers.
- **Nonlinear Modeling:** Binning is a classic method to force simpler models (like Logistic Regression or Naive Bayes) to capture complex, non-monotonic behaviors without requiring polynomial expansions.
- **Production Rules:** Never bin on your entire dataset before splitting. Always encapsulate your binning logic inside a Scikit-Learn Pipeline using `KBinsDiscretizer` to avoid data leakage.

In [ ]:
print("---------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully learned and implemented Discretization and Binning strategies!")
print("---------------------------------------------------")